# 1. Business Scenario:
A Bank has a portfolio of 50000 loans distributed across 10 economic sectors.<br>
Now, the **Risk Management Department** wants to:

**1.** Identify borrowers with a high probability of default.<br>
**2.** Estimate potential Financial Losses.<br>
**3.** Understand which industries carry the highest Credit Risk.<br>
**4.** Identify factors influencing borrowers credit ratings.<br>
**5.** Evaluate how the economic stress affects the loan portfolio.<br>
**6.** Develop a Machine Learning-based Credit Risk monitoring system.<br>

**Using Basel III formula :** EL = PD * LGD * EAD

# The given Dataset

| File | Description | Purpose |
|---|---|---|
| loan_portfolio.csv | 50,000 loans, 24 attributes | Main modelling dataset |
| credit_ratings.csv | Annual credit-rating migrations | Rating transition analysis |
| macro_stress_scenarios.csv | Macroeconomic stress scenarios | Stress testing |
| payment_history.csv | Customers' payment history & outstandings | Monthly portfolio risk |
| payment_history_dictionary.csv | Column descriptions | Data dictionary of payment csv |
| portfolio_metrics.csv | Monthly portfolio risk metrics | Portfolio monitoring |
| vintage_analysis.csv | Cumulative default curves | Vintage analysis |

**Project Architecture**

             RAW DATA
                 |
                 |
       ---------------------
       |    |    |    |    |
      Loan Ratings Macro Portfolio Vintage
      Data  Data   Data  Data   Data
       |    |    |    |    |
       ---------------------


            Loan Dataset
                 |
          DATA PROFILING
                 |
                 |
          DATA CLEANING
                 |
                 |
           DATA MERGING
                 |
                 |
          FEATURE ENGINEERING
                 |
                 |
        -----------------------
        |          |          |
       EDA       PD Model   LGD/EAD
        |          |        Models
        |          |          |
        -----------------------
                 |
                 |
          MODEL EVALUATION
                 |
                 |
         MODEL INTERPRETATION
                 |
                 |
          CREDIT RISK ENGINE
                 |
                 |
          BUSINESS INSIGHTS
                 |
                 |
          DASHBOARD / REPORT

# **2. Project Setup and Loading the dataset**

In [1]:
import os

# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical Analysis
from scipy import stats

# Machine Learning
from sklearn.model_selection import(
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import(
    LabelEncoder,
    StandardScaler,
    OneHotEncoder,
    LabelBinarizer
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import(
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score
)

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import(
    RandomForestRegressor,
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.tree import DecisionTreeClassifier

# pipeline
from imblearn.pipeline import make_pipeline as make_pipeline_imb

from imblearn.over_sampling import SMOTE

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Model Interpretation
import shap

import warnings
warnings.filterwarnings('ignore')

In [2]:
# Istallation of additional libraries if needed
!pip install xgboost lightgbm catboost shap

## 4.B.1> Loading the Dataset

In [3]:
# Loading each dataset

ratings = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/credit_ratings.csv")
loan = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/loan_portfolio.csv")
macro = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/macro_stress_scenarios.csv")
payment = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/payment_history.csv")
portfolio = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/portfolio_metrics.csv")
vintage = pd.read_csv("/kaggle/input/datasets/sergionefedov/credit-risk-dataset-50k-loans-10-sectors/vintage_analysis.csv")


In [4]:
(loan.head(5))

,loan_id,origination_date,maturity_date,maturity_months,sector,loan_type,collateral,initial_rating,credit_score,ead,...,pd_annual,lgd,el,unexpected_loss,rwa,defaulted,default_date,survival_months,recovery_rate,loss_given_default
0,L000001,2017-10-01,2019-10-01,24,Technology,mortgage,secured,B,704,1035611.99,...,0.041076,0.7858,33426.80,161508.03,442905.14,0,NaN,24,NaN,NaN
1,L000002,2016-06-01,2020-06-01,48,Healthcare,term_loan,unsecured,A,702,800786.92,...,0.000887,0.7034,499.82,16771.62,6622.66,0,NaN,48,NaN,NaN
2,L000003,2023-08-01,2024-12-31,36,Real_Estate,bond,secured,BBB,699,4327864.12,...,0.002622,0.5503,6243.54,121781.72,82726.94,0,NaN,36,NaN,NaN
3,L000004,2016-04-01,2024-12-31,120,Real_Estate,term_loan,secured,B,641,3810461.19,...,0.041679,0.3541,56237.33,269661.85,745144.64,0,NaN,120,NaN,NaN
4,L000005,2015-10-01,2019-10-01,48,Energy,bond,unsecured,CCC,597,2925538.53,...,0.136071,0.6576,261778.69,659614.05,3468567.58,0,NaN,48,NaN,NaN


In [5]:
macro.head(5)

,scenario,gdp_shock_pp,unemp_shock_pp,rate_shock_pp,credit_spread_bps,sector,base_pd,stressed_pd,pd_uplift_pp,pd_multiplier,base_lgd,stressed_lgd,total_ead,expected_loss_base,expected_loss_stress,el_increase_pct
0,baseline,0.0,0.0,0.0,0,Financials,0.022308,0.022308,0.0,1.0,0.45,0.45,3.016669e+10,3.161070e+08,3.028334e+08,-4.20
1,baseline,0.0,0.0,0.0,0,Real_Estate,0.022656,0.022656,0.0,1.0,0.40,0.40,2.530543e+10,2.617922e+08,2.293309e+08,-12.40
2,baseline,0.0,0.0,0.0,0,Consumer,0.021524,0.021524,0.0,1.0,0.55,0.55,5.049910e+09,6.904421e+07,5.978082e+07,-13.42
3,baseline,0.0,0.0,0.0,0,Industrials,0.021915,0.021915,0.0,1.0,0.50,0.50,1.571215e+10,1.963758e+08,1.721656e+08,-12.33
4,baseline,0.0,0.0,0.0,0,Technology,0.023575,0.023575,0.0,1.0,0.60,0.60,1.277305e+10,2.140718e+08,1.806719e+08,-15.60


In [6]:
payment.head(5)

,loan_id,month_on_book,payment_date,scheduled_payment,actual_payment,outstanding_balance,interest_component,principal_component,dpd,dpd_bucket,missed_payment,cumulative_missed,utilization
0,L000003,1,2023-08-01,51808.14,52711.92,4290894.80,15742.61,36969.32,0,current,0,0,NaN
1,L000003,2,2023-09-01,51808.14,51859.18,4254643.75,15608.13,36251.05,0,current,0,0,NaN
2,L000003,3,2023-10-01,51808.14,51457.49,4218662.53,15476.27,35981.22,0,current,0,0,NaN
3,L000003,4,2023-11-01,51808.14,50825.00,4183182.91,15345.38,35479.62,0,current,0,0,NaN
4,L000003,5,2023-12-01,51808.14,52244.67,4146154.57,15216.33,37028.34,0,current,0,0,NaN


## 4.B.2> Understanding the dataset<br>
Understanding the structure of the dataset before merging

In [7]:

datasets = {
    "ratings": ratings,
    "loan": loan,
    "macro": macro,
    "payment": payment,
    "portfolio": portfolio,
    "vintage": vintage
}
for name, df in datasets.items():
    print("\n", "="*60)
    print("DATASET:", name)
    
    print("Dataset Size:", df.shape)  # to get size i.e., rows,columns from each dataset
    
    print("\nColumns:")
    print(df.columns.tolist())  # to get columns from each dataset
    
    print("\nData Info:")
    print(df.info())  #  to get dtypes from each dataset

    print("\nMissing Values:")
    print(df.isnull().sum())  # to get the total null values from each dataset

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())  # to get the duplicate values from each dataset




DATASET: ratings
Dataset Size: (17939, 9)

Columns:
['issuer_id', 'sector', 'year', 'from_rating', 'to_rating', 'upgraded', 'downgraded', 'defaulted', 'notches_moved']

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17939 entries, 0 to 17938
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   issuer_id      17939 non-null  object
 1   sector         17939 non-null  object
 2   year           17939 non-null  int64 
 3   from_rating    17939 non-null  object
 4   to_rating      17939 non-null  object
 5   upgraded       17939 non-null  int64 
 6   downgraded     17939 non-null  int64 
 7   defaulted      17939 non-null  int64 
 8   notches_moved  17939 non-null  int64 
dtypes: int64(5), object(4)
memory usage: 1.2+ MB
None

Missing Values:
issuer_id        0
sector           0
year             0
from_rating      0
to_rating        0
upgraded         0
downgraded       0
defaulted        0
notches_moved 

**Getting Summary Statistics**<br>
Here we get summary statistics for numerical columns

In [8]:
datasets = {
    "ratings": ratings,
    "loan": loan,
    "macro": macro,
    "payment": payment,
    "portfolio": portfolio,
    "vintage": vintage
}
for name, df in datasets.items():
    print("\n", "="*60)
    print("DATASET:", name)

    print("\nSummary Statistics:")
    print(df.describe())


DATASET: ratings

Summary Statistics:
               year      upgraded    downgraded     defaulted  notches_moved
count  17939.000000  17939.000000  17939.000000  17939.000000   17939.000000
mean    2019.309047      0.062044      0.069123      0.023245       0.057082
std        2.871778      0.241241      0.253670      0.150686       0.552997
min     2015.000000      0.000000      0.000000      0.000000      -6.000000
25%     2017.000000      0.000000      0.000000      0.000000       0.000000
50%     2019.000000      0.000000      0.000000      0.000000       0.000000
75%     2022.000000      0.000000      0.000000      0.000000       0.000000
max     2024.000000      1.000000      1.000000      1.000000       7.000000

DATASET: loan

Summary Statistics:
       maturity_months  credit_score           ead   coupon_rate  \
count     50000.000000  50000.000000  5.000000e+04  50000.000000   
mean         53.970480    714.982560  3.298605e+06      3.079087   
std          26.716251     5

# Data Preparation

In [9]:
# Standardize date-time and year columns
loan['origination_date'] = pd.to_datetime(loan['origination_date'])
loan['maturity_date'] = pd.to_datetime(loan['maturity_date'])
loan['default_date'] = pd.to_datetime(loan['default_date'])
payment['payment_date'] = pd.to_datetime(payment['payment_date'])

**Questions we need to answer**
**For the loan dataset:**

- Is each loan_id unique?
- Are there missing loan amounts?
- Are there duplicate loans?
- What are the numerical variables?
- What are the categorical variables?

**For the payment dataset:**

- Does each loan have multiple payment records?
- Are payment dates available?
- Is there a payment status?
- Are there missed payments?
- Are there partial payments?
- Is there an overdue amount?
These answers will determine our feature engineering strategy.

# **5. Merge the Datasets**

I would use star-schema inspired data architecture.<br>
**Proposed Structure:** <br>
**Fact Table:** *loan_portfolio*<br>

**Supporting Tables:** <br>
* Credit Ratings
* Macroeconomic scenarios
* Portfolio Metrics
* Vintage analysis